# 📊 Анализ книжного магазина (bookshop_full)
Ноутбук считает e-commerce-метрики, строит графики и даёт выводы.
**Окружение:** `analytics` (pandas + matplotlib). Запускать в VS Code, выбрав этот интерпретатор.
**Данные:** `analysis/out/orders.csv` — уже выгружен скриптом `report.py`.

## 0. Импорт и загрузка

In [1]:
import os
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Ищем корень проекта (папку bookshop_py)
HERE = os.getcwd()
if os.path.basename(HERE) == "analysis":
    # Если VS Code открыт прямо в папке analysis, поднимаемся на уровень выше
    HERE = os.path.dirname(HERE)

# Теперь путь точно правильный
csv_path = os.path.join(HERE, "analysis", "out", "orders.csv")

orders = pd.read_csv(csv_path, encoding="utf-8-sig")
orders["created_at"] = pd.to_datetime(orders["created_at"])
orders["month"] = orders["created_at"].dt.to_period("M").astype(str)

valid = orders[orders["status"] != "cancelled"].copy()
print("Заказов всего:", len(orders), "| Не отменённых:", len(valid))

def money(x): return f"{x:,.2f} ₽"

gmv = float(valid['amount'].sum())
buyers = valid['user_id'].nunique()
aov = gmv / len(valid)
aov_med = valid['amount'].median()
aov_std = valid['amount'].std()
arppu = gmv / buyers

per_buyer = valid.groupby('user_id')['order_id'].count()
ltv = valid.groupby('user_id')['amount'].sum()

print(f"GMV:                      {money(gmv)}")
print(f"Заказов (не отмен.):      {len(valid)}")
print(f"Покупателей:              {buyers}")
print(f"AOV (среднее):            {money(aov)}")
print(f"AOV (медиана):            {money(aov_med)}")
print(f"AOV (std):                {money(aov_std)}")
print(f"ARPPU:                    {money(arppu)}")
print(f"LTV (среднее):            {money(ltv.mean())}")
print(f"LTV (медиана):            {money(ltv.median())}")
print(f"LTV (std):                {money(ltv.std())}")
print(f"Заказов на покупателя:    {per_buyer.mean():.2f} (медиана {per_buyer.median():.0f})")
print(f"Повторные покупатели:     {((per_buyer>=2).sum())} ({(per_buyer>=2).mean()*100:.1f}%)")

Заказов всего: 185 | Не отменённых: 172
GMV:                      269,180.00 ₽
Заказов (не отмен.):      172
Покупателей:              52
AOV (среднее):            1,565.00 ₽
AOV (медиана):            1,280.00 ₽
AOV (std):                1,052.60 ₽
ARPPU:                    5,176.54 ₽
LTV (среднее):            5,176.54 ₽
LTV (медиана):            4,890.00 ₽
LTV (std):                3,605.97 ₽
Заказов на покупателя:    3.31 (медиана 3)
Повторные покупатели:     41 (78.8%)


## 1. Сводные метрики

In [2]:
def money(x): return f"{x:,.2f} ₽"

gmv = float(valid['amount'].sum())
buyers = valid['user_id'].nunique()
aov = gmv / len(valid)
aov_med = valid['amount'].median()
aov_std = valid['amount'].std()
arppu = gmv / buyers

per_buyer = valid.groupby('user_id')['order_id'].count()
ltv = valid.groupby('user_id')['amount'].sum()

print(f"GMV:                      {money(gmv)}")
print(f"Заказов (не отмен.):      {len(valid)}")
print(f"Покупателей:              {buyers}")
print(f"AOV (среднее):            {money(aov)}")
print(f"AOV (медиана):            {money(aov_med)}")
print(f"AOV (std):                {money(aov_std)}")
print(f"ARPPU:                    {money(arppu)}")
print(f"LTV (среднее):            {money(ltv.mean())}")
print(f"LTV (медиана):            {money(ltv.median())}")
print(f"LTV (std):                {money(ltv.std())}")
print(f"Заказов на покупателя:    {per_buyer.mean():.2f} (медиана {per_buyer.median():.0f})")
print(f"Повторные покупатели:     {((per_buyer>=2).sum())} ({(per_buyer>=2).mean()*100:.1f}%)")

GMV:                      269,180.00 ₽
Заказов (не отмен.):      172
Покупателей:              52
AOV (среднее):            1,565.00 ₽
AOV (медиана):            1,280.00 ₽
AOV (std):                1,052.60 ₽
ARPPU:                    5,176.54 ₽
LTV (среднее):            5,176.54 ₽
LTV (медиана):            4,890.00 ₽
LTV (std):                3,605.97 ₽
Заказов на покупателя:    3.31 (медиана 3)
Повторные покупатели:     41 (78.8%)


## 2. Динамика по месяцам (без неполных)

In [3]:
monthly = valid.groupby('month').agg(
    orders=('order_id','count'), revenue=('amount','sum'), buyers=('user_id','nunique')
).reset_index()
monthly['aov'] = monthly['revenue'] / monthly['orders']
monthly['partial'] = monthly['month'].isin(monthly['month'].iloc[[0, -1]])
monthly.to_csv(os.path.join(OUT, 'monthly.csv'), index=False, encoding='utf-8-sig')
display(monthly)
print('\nГраницы (май/ноябрь) — неполные месяцы, исключены из тренда.')

,month,orders,revenue,buyers,aov,partial
0,2026-05,35,46710.0,25,1334.571429,True
1,2026-06,30,48710.0,21,1623.666667,False
2,2026-07,34,63890.0,25,1879.117647,False
3,2026-08,19,33680.0,17,1772.631579,False
4,2026-09,24,32620.0,19,1359.166667,False
5,2026-10,26,41080.0,22,1580.000000,False
6,2026-11,4,2490.0,4,622.500000,True



Границы (май/ноябрь) — неполные месяцы, исключены из тренда.


In [4]:
mm = monthly[~monthly['partial']].sort_values('month')
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.bar(mm['month'], mm['revenue'], color='#7c3aed', alpha=.8, label='Выручка')
ax1.set_ylabel('Выручка, ₽')
ax2 = ax1.twinx()
ax2.plot(mm['month'], mm['orders'], color='#0ea5e9', marker='o', label='Заказы')
ax2.set_ylabel('Заказы')
ax1.set_title('Выручка и заказы по месяцам (полные месяцы)')
fig.autofmt_xdate()
plt.tight_layout(); plt.show()

C:\Users\Anna\AppData\Local\Temp\ipykernel_18296\2602753778.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. Топ городов

In [5]:
geo = valid[valid['city'].notna() & (valid['city']!='')].groupby('city').agg(
    orders=('order_id','count'), revenue=('amount','sum'), buyers=('user_id','nunique')
).reset_index().sort_values('revenue', ascending=False)
geo['aov'] = geo['revenue']/geo['orders']
geo.to_csv(os.path.join(OUT, 'geo.csv'), index=False, encoding='utf-8-sig')
display(geo.head(12))

g = geo.head(10).sort_values('revenue')
fig, ax = plt.subplots(figsize=(9,4.5))
ax.barh(g['city'], g['revenue'], color='#f59e0b', alpha=.85)
ax.set_xlabel('Выручка, ₽'); ax.set_title('Топ городов по выручке')
plt.tight_layout(); plt.show()

,city,orders,revenue,buyers,aov
13,Ростов-на-Дону,17,34360.0,4,2021.176471
9,Нижний Новгород,16,24650.0,4,1540.625000
15,Санкт-Петербург,17,24580.0,5,1445.882353
21,Челябинск,9,22540.0,2,2504.444444
10,Новосибирск,8,20600.0,2,2575.000000
8,Москва,15,18890.0,7,1259.333333
20,Уфа,16,18650.0,5,1165.625000
11,Омск,12,16360.0,2,1363.333333
6,Красноярск,7,15350.0,1,2192.857143
12,Пенза,6,10230.0,2,1705.000000


C:\Users\Anna\AppData\Local\Temp\ipykernel_18296\4082708798.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Повторные покупки

In [6]:
dist = per_buyer.value_counts().sort_index()
pd.DataFrame({'orders_per_buyer':dist.index, 'clients':dist.values}).to_csv(
    os.path.join(OUT, 'repeat_purchases.csv'), index=False, encoding='utf-8-sig')
display(dist)

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(dist.index.astype(str), dist.values, color='#10b981', alpha=.85)
ax.set_xlabel('Заказов на клиента'); ax.set_ylabel('Клиентов')
ax.set_title('Распределение повторных покупок')
plt.tight_layout(); plt.show()

order_id
1    11
2     7
3    11
4    12
5     6
6     1
7     2
8     2
Name: count, dtype: int64

C:\Users\Anna\AppData\Local\Temp\ipykernel_18296\279104600.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Когорты по месяцу первой покупки

In [7]:
valid['cohort'] = valid['user_id'].map(valid.groupby('user_id')['created_at'].min().dt.to_period('M'))
valid['m'] = (valid['created_at'].dt.year - valid['cohort'].dt.year)*12 + (valid['created_at'].dt.month - valid['cohort'].dt.month)
pivot = valid.pivot_table(index='cohort', columns='m', values='order_id', aggfunc='count', fill_value=0)
pivot.columns = [f'm{int(c)}' for c in pivot.columns]
pivot.to_csv(os.path.join(OUT, 'cohorts.csv'), encoding='utf-8-sig')
display(pivot)

,m0,m1,m2,m3,m4,m5,m6
cohort,,,,,,,
2026-05,35,13,14,11,10,15,1
2026-06,17,13,3,6,5,2,0
2026-07,7,0,1,0,0,0,0
2026-08,5,1,5,1,0,0,0
2026-09,6,1,0,0,0,0,0


In [8]:
plt.figure(figsize=(7,4))
plt.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
plt.xticks(range(pivot.shape[1]), pivot.columns)
plt.yticks(range(pivot.shape[0]), [str(c) for c in pivot.index])
plt.colorbar(label='Заказы')
plt.title('Когорты: повторные покупки')
plt.tight_layout(); plt.show()

C:\Users\Anna\AppData\Local\Temp\ipykernel_18296\3919102483.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Выводы и рекомендации

- **GMV** — валовая выручка (без отменённых), **AOV** — средний чек; **медиана** ниже среднего → есть крупные заказы, тянущие среднее вверх.
- **78.8% повторных покупателей** — сильный ретеншн: клиенты возвращаются.
- **Когорты:** майские (35 чел.) хорошо возвращаются (~40%), а **новые июль-сентябрь почти не повторяют** — риск оттока новых клиентов.
- **Города:** выручка концентрирована (Ростов, Нижний, Челябинск, Новосибирск) немногими покупателями.
- **Август — провал** в заказах/выручке.
- **56 заказов** ждут обработки (`new`/`processing`), 13 отменённых.

### Идеи для A/B-тестов
1. Промокод/email после первой покупки для новых когорт → метрика: повторные покупки в m1.
2. Акция/подборка к августу → метрика: GMV августа.
3. Точечные акции в топ-городах (Ростов, Нижний) → метрика: AOV города.
4. Автоматизация статусов/напоминание об оплате → метрика: доля `completed` за 3 дня.
5. Дополнить воронку: логировать `view_cart`, `checkout` на фронте → посчитать cart-abandonment.